# Notebook 3: ALFF Node Features

Extracts per-region mALFF values from the voxelwise ALFF/mALFF maps (computed via DPARSFA across 3 bands: classical, slow-5, slow-4) into the format `datasets/Dataset.py` expects.

Uses the **same atlas and method as PCC** (`aal_mask_pad.nii.gz`, FLIRT identity+trilinear resample into atlas space, labels < 9001 = 90 regions) -- validated on real data (r=1.0000 vs official `rois_aal`, and the FLIRT-on-3D-statistical-map procedure separately validated on one subject before this full run).

- mALFF is the primary feature (raw ALFF is not saved -- decided, not deferred).
- Output: `{FILE_ID}_nf.mat`, key `norm_matrix`, shape (90, 3), z-scored per subject per band, UNHARMONIZED.
- Routed by `DX_GROUP`: ASD -> `ASD_NF/` (Dataset.py assigns y=1), control -> `NC_NF/` (y=0).

In [1]:
import subprocess
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy.io import savemat

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PHENOTYPIC_PATH = DATA_DIR / "phenotypic_filtered_v2.csv"

DPARSF_WORK = PROJECT_ROOT / "data" / "dparsf_work"
ATLAS_PATH = PROJECT_ROOT / "data" / "software" / "atlases" / "aal_mask_pad.nii.gz"
FSL_SIF = PROJECT_ROOT / "data" / "software" / "containers" / "fsl_6.0.7.4.sif"
IDENT_MAT = DATA_DIR / "_validation" / "ident.mat"

DATASET_ROOT = PROJECT_ROOT / "data" / "GraSTIACL_ABIDE_979" / "raw"
ASD_NF_DIR = DATASET_ROOT / "ASD_NF"
NC_NF_DIR = DATASET_ROOT / "NC_NF"
ASD_NF_DIR.mkdir(parents=True, exist_ok=True)
NC_NF_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = DPARSF_WORK / "_nf_tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

BANDS = ["slow5", "slow4", "classical"]
AAL90_LABEL_CUTOFF = 9001
COVERAGE_MIN_FRACTION = 0.5
OUTLIER_SD_THRESHOLD = 5.0

## Step 1: Load phenotypic_filtered_v2.csv (956 subjects)

In [2]:
df = pd.read_csv(PHENOTYPIC_PATH)
print(f"Subjects: {len(df)}")
assert len(df) == 956
df[["FILE_ID", "SITE_ID", "DX_GROUP"]].head()

Subjects: 956


,FILE_ID,SITE_ID,DX_GROUP
0,Pitt_0050003,PITT,1
1,Pitt_0050004,PITT,1
2,Pitt_0050005,PITT,1
3,Pitt_0050006,PITT,1
4,Pitt_0050007,PITT,1


## Step 2: Map subject -> chunk dir, per band; load atlas region labels

In [3]:
def build_subject_to_chunk(band: str) -> dict:
    mapping = {}
    for i in range(1, 11):
        chunk = f"{i:02d}"
        subj_file = DPARSF_WORK / f"full_{band}_chunk{chunk}" / "SubjectList.txt"
        for sid in subj_file.read_text().splitlines():
            sid = sid.strip()
            if sid:
                mapping[sid] = chunk
    return mapping


subject_to_chunk = {band: build_subject_to_chunk(band) for band in BANDS}
for band in BANDS:
    print(f"{band}: {len(subject_to_chunk[band])} subjects mapped")

atlas_img = nib.load(str(ATLAS_PATH))
atlas_data = np.round(atlas_img.get_fdata()).astype(int)
region_labels = sorted(int(l) for l in np.unique(atlas_data) if l != 0 and l < AAL90_LABEL_CUTOFF)
assert len(region_labels) == 90

expected_voxel_counts = np.array([(atlas_data == label).sum() for label in region_labels])
print(f"\nAtlas: {len(region_labels)} regions, expected voxel counts range {expected_voxel_counts.min()}-{expected_voxel_counts.max()}")

slow5: 979 subjects mapped
slow4: 979 subjects mapped
classical: 979 subjects mapped

Atlas: 90 regions, expected voxel counts range 67-2087


## Steps 3-4: FLIRT resample into atlas grid, average voxels per region (956 x 3 bands)

3.5 was already validated separately (1 subject, 1 band, before this loop): grids/affines matched, all 90 regions present, physiologically plausible values.

In [4]:
def flirt_resample_to_atlas(malff_path: Path, out_path: Path) -> None:
    cmd = [
        "apptainer", "exec", "--bind", "/mnt/scratch:/mnt/scratch", str(FSL_SIF),
        "flirt",
        "-in", str(malff_path),
        "-ref", str(ATLAS_PATH),
        "-out", str(out_path),
        "-applyxfm",
        "-init", str(IDENT_MAT),
        "-interp", "trilinear",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"FLIRT failed for {malff_path}: {result.stderr}")


n_subjects = len(df)
n_bands = len(BANDS)
malff_region_matrix = np.full((n_subjects, 90, n_bands), np.nan, dtype=np.float64)
valid_voxel_count_matrix = np.zeros((n_subjects, 90, n_bands), dtype=np.int32)
missing = []

for si, file_id in enumerate(df["FILE_ID"]):
    for bi, band in enumerate(BANDS):
        chunk = subject_to_chunk[band][file_id]
        malff_path = DPARSF_WORK / f"full_{band}_chunk{chunk}" / "Results" / "ALFF_FunImgD" / f"mALFFMap_{file_id}.nii"
        if not malff_path.exists():
            missing.append((file_id, band))
            continue

        resampled_path = TMP_DIR / f"{file_id}_{band}_in_atlas.nii.gz"
        flirt_resample_to_atlas(malff_path, resampled_path)

        resampled_data = nib.load(str(resampled_path)).get_fdata()
        resampled_path.unlink()  # clean up as we go, 956x3 files would be a lot to keep

        # coverage diagnostic only (not used to filter the mean below) -- a voxel counts as
        # "valid" if finite and nonzero; DPARSFA's own brain mask legitimately zeroes some
        # region-boundary voxels, so this is tracked for QC, not excluded from the average
        valid_mask = np.isfinite(resampled_data) & (resampled_data != 0)

        for ri, label in enumerate(region_labels):
            region_mask = atlas_data == label
            valid_voxel_count_matrix[si, ri, bi] = (region_mask & valid_mask).sum()
            # mean over ALL region voxels unconditionally -- matches the validated
            # rois_aal/Global-PCC method and the single-subject 3.5 validation
            malff_region_matrix[si, ri, bi] = resampled_data[region_mask].mean()

    if (si + 1) % 100 == 0:
        print(f"  {si + 1}/{n_subjects} subjects done")

print(f"\nDone. Missing: {len(missing)}")
if missing:
    print(missing[:10])

  100/956 subjects done


  200/956 subjects done


  300/956 subjects done


  400/956 subjects done


  500/956 subjects done


  600/956 subjects done


  700/956 subjects done


  800/956 subjects done


  900/956 subjects done



Done. Missing: 0


## Step 6: QC -- coverage >= 50%, finite, exact-zero, and >5SD-from-median outlier flags

Then compare flagged subjects against the PCC dead-ROI list (23 subjects) -- expected to differ, since the two checks use different underlying data (rois_aal time-series variance vs. ALFF voxel coverage/values).

In [5]:
coverage_fraction = valid_voxel_count_matrix / expected_voxel_counts[None, :, None]

flag_low_coverage = coverage_fraction < COVERAGE_MIN_FRACTION
flag_nonfinite = ~np.isfinite(malff_region_matrix)
flag_exact_zero = malff_region_matrix == 0.0

# >5 SD from cohort median, per region per band (using only finite values)
region_median = np.nanmedian(malff_region_matrix, axis=0)  # (90, n_bands)
region_std = np.nanstd(malff_region_matrix, axis=0)  # (90, n_bands)
with np.errstate(invalid="ignore"):
    flag_outlier = np.abs(malff_region_matrix - region_median[None, :, :]) > (OUTLIER_SD_THRESHOLD * region_std[None, :, :])
flag_outlier = flag_outlier & np.isfinite(malff_region_matrix)

any_flag = flag_low_coverage | flag_nonfinite | flag_exact_zero | flag_outlier
subject_any_flag = any_flag.any(axis=(1, 2))
subject_flag_counts = any_flag.sum(axis=(1, 2))

alff_qc_df = pd.DataFrame({
    "FILE_ID": df["FILE_ID"].values,
    "n_low_coverage": flag_low_coverage.sum(axis=(1, 2)),
    "n_nonfinite": flag_nonfinite.sum(axis=(1, 2)),
    "n_exact_zero": flag_exact_zero.sum(axis=(1, 2)),
    "n_outlier": flag_outlier.sum(axis=(1, 2)),
    "any_flag": subject_any_flag,
    "total_flags": subject_flag_counts,
})

print(f"Subjects with >=1 flag: {subject_any_flag.sum()} / {n_subjects}")
print(alff_qc_df[alff_qc_df["any_flag"]].sort_values("total_flags", ascending=False).to_string(index=False))

alff_flagged_ids = set(alff_qc_df.loc[alff_qc_df["any_flag"], "FILE_ID"])
# dead_roi_ids from notebook 02 isn't in-memory here; load from the persisted QC file instead
pcc_qc = pd.read_csv(DATA_DIR / "global_pcc_qc.csv")
pcc_dead_roi_ids = set(pcc_qc.loc[~pcc_qc["finite"], "FILE_ID"])

overlap = alff_flagged_ids & pcc_dead_roi_ids
alff_only = alff_flagged_ids - pcc_dead_roi_ids
pcc_only = pcc_dead_roi_ids - alff_flagged_ids

print(f"\nPCC dead-ROI subjects (23, already dropped from this 956-cohort): {len(pcc_dead_roi_ids)}")
print(f"ALFF-flagged subjects (from this 956-cohort): {len(alff_flagged_ids)}")
print(f"Overlap (should be 0, since PCC's 23 are already excluded from df): {len(overlap)}")
print(f"ALFF-only flags (new subjects, not in PCC's dead-ROI list): {len(alff_only)}")
if alff_only:
    print(sorted(alff_only))

Subjects with >=1 flag: 956 / 956
         FILE_ID  n_low_coverage  n_nonfinite  n_exact_zero  n_outlier  any_flag  total_flags
    Yale_0050605              69            0             0          7      True           73
    SDSU_0050204              66            0             0          3      True           69
Leuven_2_0050736              51            0             0          9      True           58
 Caltech_0051462              54            0             0          0      True           54
 Caltech_0051468              51            0             0          1      True           52
 Caltech_0051473              48            0             3          3      True           51
 Caltech_0051456              51            0             0          0      True           51
 Caltech_0051458              48            0             0          0      True           48
    SDSU_0050211              39            0             0         10      True           47
 Caltech_0051472          

## Step 7: z-score per subject per band

In [6]:
# z-score independently per subject, per band, across that subject's 90 regions
# (NaN regions stay NaN here -- Dataset.py's own np.nan_to_num() zeroes them at load time,
# same handling as the ADJ/cropped_matrix path)
subject_band_mean = np.nanmean(malff_region_matrix, axis=1, keepdims=True)  # (n_subjects, 1, n_bands)
subject_band_std = np.nanstd(malff_region_matrix, axis=1, keepdims=True)  # (n_subjects, 1, n_bands)

norm_matrix_all = (malff_region_matrix - subject_band_mean) / subject_band_std

print(f"norm_matrix_all shape: {norm_matrix_all.shape}")
print(f"Finite entries: {np.isfinite(norm_matrix_all).sum()} / {norm_matrix_all.size}")
print(f"Per-subject-band mean should be ~0, std ~1 (ignoring NaN):")
print(f"  mean of means: {np.nanmean(np.nanmean(norm_matrix_all, axis=1)):.6f}")
print(f"  mean of stds:  {np.nanmean(np.nanstd(norm_matrix_all, axis=1)):.6f}")

norm_matrix_all shape: (956, 90, 3)
Finite entries: 258120 / 258120
Per-subject-band mean should be ~0, std ~1 (ignoring NaN):
  mean of means: -0.000000
  mean of stds:  1.000000


## Step 8: Route by DX_GROUP, save {FILE_ID}_nf.mat (key `norm_matrix`, unharmonized)

ASD (DX_GROUP==1) -> `ASD_NF/` (Dataset.py assigns y=1); control (DX_GROUP==2) -> `NC_NF/` (y=0).

In [7]:
for si, row in df.iterrows():
    file_id = row["FILE_ID"]
    dx_group = int(row["DX_GROUP"])
    out_dir = ASD_NF_DIR if dx_group == 1 else NC_NF_DIR

    norm_matrix = norm_matrix_all[si].astype(np.float64)  # (90, 3): slow5, slow4, classical
    savemat(out_dir / f"{file_id}_nf.mat", {"norm_matrix": norm_matrix})

n_asd_files = len(list(ASD_NF_DIR.glob("*_nf.mat")))
n_nc_files = len(list(NC_NF_DIR.glob("*_nf.mat")))

print(f"ASD_NF: {n_asd_files} (expect {(df['DX_GROUP'] == 1).sum()})")
print(f"NC_NF:  {n_nc_files} (expect {(df['DX_GROUP'] == 2).sum()})")
assert n_asd_files == (df["DX_GROUP"] == 1).sum()
assert n_nc_files == (df["DX_GROUP"] == 2).sum()
assert n_asd_files + n_nc_files == len(df)

alff_qc_df.to_csv(DATA_DIR / "alff_nf_qc.csv", index=False)
print(f"\nQC saved to {DATA_DIR / 'alff_nf_qc.csv'}")
print("Band order in norm_matrix columns:", BANDS)

ASD_NF: 455 (expect 455)
NC_NF:  501 (expect 501)

QC saved to /users/3171356m/muhammad/GraSTIACL/data/raw/alff_nf_qc.csv
Band order in norm_matrix columns: ['slow5', 'slow4', 'classical']
